# First prototype for HyperBubble Resolution-Oriented GNN

In [1]:
from __future__ import annotations
import argparse
import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

In [2]:
NUC2ID = {'A': 1, 'C': 2, 'G': 3, 'T': 4, 'N': 5}
ID2NUC = {v: k for k, v in NUC2ID.items()}

def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v


In [3]:
class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from JSONL:
      - Node features:
          seq_tokens : [N, K] Long
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 5] Float (len_nodes, len_bp, cov_min, cov_mean, on_ref)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask    : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "inside_nodes", []):
            if isinstance(n, dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov", 0))

        ensure_node(start_seq)
        ensure_node(end_seq)

        edge_src, edge_dst, edge_attr = [], [], []
        for e in _safe_get(rec, "edges", []):
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u); edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0)),
                1.0 if e.get("on_ref", False) else 0.0
            ])

        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K] Long
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1] Float

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0, 5), dtype=torch.float32))

        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,
            x_cov=x_cov,
            edge_index=edge_index,
            edge_attr=edge_attr,
            start_idx=start_idx,
            end_idx=end_idx,
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,
            label_path_idx=torch.tensor(label_path_idx, dtype=torch.long),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])

In [4]:
class ScriptableDenseGCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H: torch.Tensor, edge_index_local: torch.Tensor, n_nodes: int) -> torch.Tensor:
        # identical math to your DenseGCNLayer but without try/except
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src = edge_index_local[0]
            dst = edge_index_local[1]
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops (TorchScript-safe)
        flat = A.view(-1)
        diag_idx = torch.arange(0, n_nodes * n_nodes, step=n_nodes + 1, device=H.device)
        flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
        A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        d = deg.pow(-0.5)
        A_norm = (d[:, None] * A) * d[None, :]
        return A_norm @ self.lin(H)

class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,
        embed_dim=16,
        gcn_hidden=32,
        edge_hidden=16,
        edge_feat_dim=5,
        use_lstm=False,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=False)
            self.node_in = gcn_hidden + 1
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = ScriptableDenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = ScriptableDenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)
        X = torch.cat([Hseq, x_cov], dim=1)
        return X

    def forward(self, data):
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        out_node = X0.new_zeros((N, self.gcn_hidden))
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            out_node[node_ids] = H

        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]
        V = out_node[v]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits

In [5]:
USE_DIRECTML = True

import torch
device = torch.device('cpu')
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device('cuda')
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device('cpu')

import torch.nn as nn
import torch.nn.functional as F

class DenseGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)

    def forward(self, H, edge_index_local, n_nodes: int):
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src, dst = edge_index_local
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops w/o torch.eye
        idx = torch.arange(n_nodes, device=H.device)
        try:
            A[idx, idx] += 1
        except Exception:
            flat = A.view(-1)
            diag_idx = torch.arange(0, n_nodes*n_nodes, step=n_nodes+1, device=H.device)
            flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
            A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        D_inv_sqrt = deg.pow(-0.5)
        A_norm = (D_inv_sqrt[:, None] * A) * D_inv_sqrt[None, :]
        return A_norm @ self.lin(H)

# ---- Simple GNN with sequence embedding + GCN + edge MLP ----
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=5,          # tokens: 0=PAD, A,C,G,T (and optionally N)
        embed_dim=32,          # per-token embedding dim
        gcn_hidden=64,         # node hidden dim (consistent throughout)
        edge_hidden=64,        # hidden in edge MLP
        edge_feat_dim=5,       # [len_nodes, len_bp, cov_min, cov_mean, on_ref]
        use_lstm=False,        # optional: token encoder = mean or BiLSTM
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=False)
            self.node_in = gcn_hidden + 1   # + coverage scalar
        else:
            self.node_in = embed_dim + 1

        self.gcn_hidden = gcn_hidden
        self.gcn1 = DenseGCNLayer(self.node_in, gcn_hidden)
        self.gcn2 = DenseGCNLayer(gcn_hidden, gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        """
        seq_tokens: [N, K] tokenized k-mer
        x_cov     : [N, 1]
        returns   : [N, node_in]
        """
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            mean_init = (E * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.clamp(min=1).unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        X = torch.cat([Hseq, x_cov], dim=1)
        return X

    def forward(self, data):
        """
        data: PyG batch with fields:
          - seq_tokens [N,K], x_cov [N,1], edge_index [2,E], edge_attr [E,5],
            batch [N] (graph ids), num_nodes, etc.
        returns:
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
        N = data.num_nodes
        E = data.edge_index.size(1)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        out_node = X0.new_zeros((N, self.gcn_hidden))
        batch_vec = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)
        num_graphs = int(batch_vec.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch_vec == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            ei = data.edge_index
            keep = (local_map[ei[0]] >= 0) & (local_map[ei[1]] >= 0)
            keep_idx = keep.nonzero(as_tuple=False).view(-1)
            if keep_idx.numel() == 0:
                H = F.relu(self.gcn1(X0[node_ids], edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                H = F.relu(self.gcn2(H,          edge_index_local=torch.empty(2,0,device=device,dtype=torch.long), n_nodes=n_nodes))
                out_node[node_ids] = H
                continue

            src_local = local_map[ei[0, keep_idx]]
            dst_local = local_map[ei[1, keep_idx]]
            edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn1(X0[node_ids], edge_local, n_nodes))
            H = F.relu(self.gcn2(H,            edge_local, n_nodes))
            out_node[node_ids] = H

        if E == 0:
            return torch.empty((0,), device=device)

        u, v = data.edge_index
        U = out_node[u]
        V = out_node[v]
        EA = data.edge_attr if hasattr(data, "edge_attr") and data.edge_attr.numel() > 0 \
             else out_node.new_zeros((E, 5))
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.edge_mlp(feats).squeeze(-1)
        return logits


Using DirectML: privateuseone:0


In [6]:
def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])

    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

@torch.no_grad()
def eval_choice_all(model, loader, device):
    model.eval()
    total_decisions, correct_decisions = 0, 0
    total_bubbles, correct_bubbles = 0, 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue
            total_bubbles += 1

            all_correct = True
            for cand_idx, gold_pos in decisions:
                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                correct_decisions += ok
                total_decisions += 1
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    dec_acc = correct_decisions / max(total_decisions, 1)
    bub_acc = correct_bubbles / max(total_bubbles, 1)
    print(f"[choice-eval] decisions={total_decisions} acc_dec={dec_acc:.3f} | bubbles={total_bubbles} acc_bub={bub_acc:.3f}")
    return dec_acc, bub_acc

def _cpu_state_dict(sd):
    return {k: v.detach().cpu().clone() for k, v in sd.items()}

In [7]:
def gcn_forward_torchscript(H: torch.Tensor, edge_index_local: torch.Tensor, n_nodes: int, lin: nn.Linear) -> torch.Tensor:
    if n_nodes == 0:
        return H
    A = H.new_zeros((n_nodes, n_nodes))
    if edge_index_local.numel() > 0:
        src, dst = edge_index_local[0], edge_index_local[1]
        one = torch.ones_like(src, dtype=H.dtype)
        A.index_put_((src, dst), one, accumulate=True)

    flat = A.view(-1)
    diag_idx = torch.arange(0, n_nodes * n_nodes, step=n_nodes + 1, device=H.device)
    flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
    A = flat.view(n_nodes, n_nodes)

    deg = A.sum(dim=1) + 1e-8
    d = deg.pow(-0.5)
    A_norm = (d[:, None] * A) * d[None, :]
    return A_norm @ lin(H)

class TSWrapper(nn.Module):
    """
    TorchScriptable front-end:
      forward_ts(
        seq_tokens [N,K] long,
        x_cov      [N,1] float,
        edge_index [2,E] long,
        batch      [N]   long,
        edge_attr  [E,5] float (optional -> can pass empty)
      ) -> logits [E] float
    """
    def __init__(self, core_model: HyperbubbleGNN):
        super().__init__()
        self.core = core_model

    @torch.jit.export
    def forward_ts(
        self,
        seq_tokens: torch.Tensor,
        x_cov: torch.Tensor,
        edge_index: torch.Tensor,
        batch: torch.Tensor,
        edge_attr: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        device = seq_tokens.device
        N = int(seq_tokens.size(0))
        E = int(edge_index.size(1)) if edge_index.numel() > 0 else 0
        if edge_attr is None or edge_attr.numel() == 0:
            edge_attr = torch.zeros((E, 5), dtype=torch.float32, device=device)

        # encode nodes
        Eemb = self.core.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)
        lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
        Hseq = (Eemb * mask).sum(dim=1) / lengths.unsqueeze(-1)
        X0 = torch.cat([Hseq, x_cov], dim=1)

        out_node = X0.new_zeros((N, self.core.gcn_hidden))
        num_graphs = int(batch.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            node_ids = (batch == g).nonzero(as_tuple=False).view(-1)
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map[node_ids] = torch.arange(n_nodes, device=device, dtype=torch.long)

            if E == 0:
                edge_local = torch.empty((2, 0), dtype=torch.long, device=device)
            else:
                src = edge_index[0]
                dst = edge_index[1]
                keep = (local_map[src] >= 0) & (local_map[dst] >= 0)
                keep_idx = keep.nonzero(as_tuple=False).view(-1)
                if keep_idx.numel() == 0:
                    edge_local = torch.empty((2, 0), dtype=torch.long, device=device)
                else:
                    src_local = local_map[src.index_select(0, keep_idx)]
                    dst_local = local_map[dst.index_select(0, keep_idx)]
                    edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(gcn_forward_torchscript(X0.index_select(0, node_ids), edge_local, n_nodes, self.core.gcn1.lin))
            H = F.relu(gcn_forward_torchscript(H, edge_local, n_nodes, self.core.gcn2.lin))
            out_node.index_copy_(0, node_ids, H)

        if E == 0:
            return out_node.new_zeros((0,))

        u = edge_index[0].to(torch.long)
        v = edge_index[1].to(torch.long)
        U = out_node.index_select(0, u)
        V = out_node.index_select(0, v)
        EA = edge_attr.to(dtype=U.dtype)
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.core.edge_mlp(feats).squeeze(-1)
        return logits

In [8]:
MODEL_PATH = "model2.8k_upgraded_dataset.pth"
TS_OUT    = "hyperbubble_gnn.ts"
BATCH_SIZE = 64
USE_DIRECTML = True

import torch, sys
device = torch.device("cpu")
if not USE_DIRECTML and torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_DIRECTML:
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML:", device)
    except Exception as e:
        print("DirectML not available, falling back to CPU:", e)
        device = torch.device("cpu")
print("Device:", device)

TRAIN_PATHS = [
]
TEST_PATHS = [
    "lower_cov_data/klebsiella_dataset_cov20_full_upgraded_ratio_lt_35.jsonl",
]

from pathlib import Path
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader

for p in TRAIN_PATHS + TEST_PATHS:
    if not Path(p).is_file():
        raise FileNotFoundError(f"Missing: {p}")


test_full  = HyperbubbleDataset(TEST_PATHS)

def _check_k(ds):
    if len(ds) == 0:
        return
    s0 = ds[0].seq_tokens.size(1)
    for i in (len(ds)//2, len(ds)-1):
        s = ds[i].seq_tokens.size(1)
        assert s == s0, f"Inconsistent k across samples: {s0} vs {s}"
_check_k(test_full)

def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)

test_dataset  = labeled_subset(test_full)

print(f"Test  (labeled): {len(test_dataset)}")

test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Using DirectML: privateuseone:0
Device: privateuseone:0
Test  (labeled): 283


In [9]:
model = HyperbubbleGNN(
    vocab_size=5, embed_dim=16, gcn_hidden=32, edge_hidden=16, edge_feat_dim=5, use_lstm=False
).to(device)

state = torch.load(MODEL_PATH, map_location="cpu")
model.load_state_dict(state)
model.eval()

with torch.no_grad():
    dec_acc, bub_acc = eval_choice_all(model, test_loader, device)
print(f"[loaded-eval] acc_bub={bub_acc:.4f} acc_dec={dec_acc:.4f}")

C:\Users\Przemek\AppData\Local\Temp\ipykernel_18912\214899032.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(MODEL_PATH, map_location="cpu")


[choice-eval] decisions=303 acc_dec=0.690 | bubbles=283 acc_bub=0.675
[loaded-eval] acc_bub=0.6749 acc_dec=0.6898


In [10]:
import torch.nn as nn
import torch.nn.functional as F

def gcn_forward_torchscript(H: torch.Tensor, edge_index_local: torch.Tensor, n_nodes: int, lin: nn.Linear) -> torch.Tensor:
    if n_nodes == 0:
        return H
    A = H.new_zeros((n_nodes, n_nodes))
    if edge_index_local.numel() > 0:
        src, dst = edge_index_local[0], edge_index_local[1]
        one = torch.ones_like(src, dtype=H.dtype)
        A.index_put_((src, dst), one, accumulate=True)

    flat = A.view(-1)
    diag_idx = torch.arange(0, n_nodes * n_nodes, step=n_nodes + 1, device=H.device)
    flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
    A = flat.view(n_nodes, n_nodes)

    deg = A.sum(dim=1) + 1e-8
    d = deg.pow(-0.5)
    A_norm = (d[:, None] * A) * d[None, :]
    return A_norm @ lin(H)

class TSWrapper(nn.Module):
    """
    TorchScriptable front-end around HyperbubbleGNN.
    Exposes forward_ts(
        seq_tokens [N,K] long,
        x_cov      [N,1] float,
        edge_index [2,E] long,
        batch      [N]   long,
        edge_attr  [E,5] float  (optional; pass empty if none)
    ) -> logits [E] float
    """
    def __init__(self, core_model: nn.Module):
        super().__init__()
        self.core = core_model

    def gcn_forward(self, H: torch.Tensor, edge_index_local: torch.Tensor, n_nodes: int, lin: nn.Linear) -> torch.Tensor:
        if n_nodes == 0:
            return H
        A = H.new_zeros((n_nodes, n_nodes))
        if edge_index_local.numel() > 0:
            src = edge_index_local[0]
            dst = edge_index_local[1]
            one = torch.ones_like(src, dtype=H.dtype)
            A.index_put_((src, dst), one, accumulate=True)

        # add self-loops (TorchScript-safe)
        flat = A.view(-1)
        diag_idx = torch.arange(0, n_nodes * n_nodes, step=n_nodes + 1, device=H.device)
        flat.index_add_(0, diag_idx, torch.ones(n_nodes, device=H.device, dtype=H.dtype))
        A = flat.view(n_nodes, n_nodes)

        deg = A.sum(dim=1) + 1e-8
        d = deg.pow(-0.5)
        A_norm = (d[:, None] * A) * d[None, :]
        return A_norm @ lin(H)

    @torch.jit.export
    def forward_ts(
        self,
        seq_tokens: torch.Tensor,   # [N,K] long
        x_cov: torch.Tensor,        # [N,1] float
        edge_index: torch.Tensor,   # [2,E] long
        batch: torch.Tensor,        # [N]   long
        edge_attr: torch.Tensor = None  # [E,5] float or empty
    ) -> torch.Tensor:              # returns [E] float
        device = seq_tokens.device
        N = int(seq_tokens.size(0))
        E = int(edge_index.size(1)) if edge_index.numel() > 0 else 0
        if edge_attr is None or edge_attr.numel() == 0:
            edge_attr = torch.zeros((E, 5), dtype=torch.float32, device=device)

        # Node encoding: mean over token embeddings + coverage
        Eemb = self.core.embed(seq_tokens)             # [N,K,embed]
        mask = (seq_tokens != 0).float().unsqueeze(-1) # [N,K,1]
        lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
        Hseq = (Eemb * mask).sum(dim=1) / lengths.unsqueeze(-1)
        X0 = torch.cat([Hseq, x_cov], dim=1)           # [N, node_in]

        out_node = X0.new_zeros((N, self.core.gcn_hidden))
        # number of graphs in batch
        num_graphs = int(batch.max().item()) + 1 if N > 0 else 0

        for g in range(num_graphs):
            # TorchScript-safe equivalent of nonzero(as_tuple=False)
            node_ids = torch.where(batch == g)[0]
            n_nodes = int(node_ids.numel())
            if n_nodes == 0:
                continue

            local_map = torch.full((N,), -1, dtype=torch.long, device=device)
            local_map.index_copy_(0, node_ids, torch.arange(n_nodes, device=device, dtype=torch.long))

            if E == 0:
                edge_local = torch.empty((2, 0), dtype=torch.long, device=device)
            else:
                src = edge_index[0]; dst = edge_index[1]
                keep = (local_map.index_select(0, src) >= 0) & (local_map.index_select(0, dst) >= 0)
                keep_idx = torch.where(keep)[0]
                if int(keep_idx.numel()) == 0:
                    edge_local = torch.empty((2, 0), dtype=torch.long, device=device)
                else:
                    src_local = local_map.index_select(0, src.index_select(0, keep_idx))
                    dst_local = local_map.index_select(0, dst.index_select(0, keep_idx))
                    edge_local = torch.stack([src_local, dst_local], dim=0)

            H = F.relu(self.gcn_forward(X0.index_select(0, node_ids), edge_local, n_nodes, self.core.gcn1.lin))
            H = F.relu(self.gcn_forward(H, edge_local, n_nodes, self.core.gcn2.lin))
            out_node.index_copy_(0, node_ids, H)

        if E == 0:
            return out_node.new_zeros((0,), dtype=torch.float32, device=device)

        u = edge_index[0].to(torch.long)
        v = edge_index[1].to(torch.long)
        U = out_node.index_select(0, u)
        V = out_node.index_select(0, v)
        EA = edge_attr.to(dtype=U.dtype)
        feats = torch.cat([U, V, EA], dim=1)
        logits = self.core.edge_mlp(feats).squeeze(-1)
        return logits

In [12]:
def _clone_into_scriptable(src_layer: nn.Module) -> ScriptableDenseGCNLayer:
    out = ScriptableDenseGCNLayer(src_layer.lin.in_features, src_layer.lin.out_features)
    # copy weights
    out.lin.weight.data.copy_(src_layer.lin.weight.data)
    if src_layer.lin.bias is not None:
        out.lin.bias.data.copy_(src_layer.lin.bias.data)
    return out

model.gcn1 = _clone_into_scriptable(model.gcn1)
model.gcn2 = _clone_into_scriptable(model.gcn2)

wrapper = TSWrapper(model).eval()
scripted = torch.jit.script(wrapper)
scripted.save(TS_OUT)
print(f"[export] TorchScript saved to: {TS_OUT}")

RuntimeError: 
'Tensor (inferred)' object has no attribute or method 'seq_tokens'.:
  File "C:\Users\Przemek\AppData\Local\Temp\ipykernel_18912\1997271116.py", line 106
          logits [E]  (edge-wise)
        """
        device = data.seq_tokens.device
                 ~~~~~~~~~~~~~~~ <--- HERE
        N = data.num_nodes
        E = data.edge_index.size(1)
